In [1]:
from google.colab import drive
import os

# 1. Mount Drive first to ensure connection works
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

print("✅ Drive Mounted Successfully.")

Mounted at /content/drive
✅ Drive Mounted Successfully.


# Final integrated code with RAG and Agentic Control Module

In [2]:
!pip install -q transformers==4.43.3 accelerate sentence-transformers bitsandbytes
!pip install -q faiss-cpu --no-deps
print("✅ Libraries Installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 117.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 83.7 MB/s eta 0:00:00
✅ Libraries Installed.


In [3]:
import os
import pickle
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer
import faiss  # Now safe to import

# --- CONFIGURATION ---
BASE_DB_PATH = "/content/drive/My Drive/Colab_RAG_Project/chapters_db"
CHAPTERS = ["Chapter_1", "Chapter_2", "Chapter_3", "Chapter_4", "Chapter_5", "Chapter_6"]

# --- LOAD MODELS ---
print("Loading Embedding Model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print("Loading Phi-3.5...")
model_id = "microsoft/Phi-3.5-mini-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    return_full_text=False
)

# --- RAG CLASS ---
class CourseRAGTool:
    def __init__(self, base_path, chapter_list, embedder):
        self.indices = {}
        self.texts = {}
        self.embedder = embedder
        print(f"\nLoading Chapters...")
        for chap in chapter_list:
            path = os.path.join(base_path, chap)
            try:
                self.indices[chap] = faiss.read_index(os.path.join(path, "index.faiss"))
                with open(os.path.join(path, "texts.pkl"), "rb") as f:
                    self.texts[chap] = pickle.load(f)
                print(f"  [OK] {chap}")
            except Exception as e:
                print(f"  [ERR] {chap}: {e}")

    def search(self, query, top_k=2):
        query_vec = self.embedder.encode([query]).astype('float32')
        results = []
        for chap, index in self.indices.items():
            D, I = index.search(query_vec, top_k)
            for i in range(top_k):
                idx = I[0][i]
                if idx != -1:
                    results.append(f"[Source: {chap}] {self.texts[chap][idx]}")
        return "\n\n".join(results[:3])

# Initialize RAG
rag_tool = CourseRAGTool(BASE_DB_PATH, CHAPTERS, embedder)
print("\n✅ System Ready.")

Loading Embedding Model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]


Loading Chapters...
  [ERR] Chapter_1: Error in faiss::FileIOReader::FileIOReader(const char*) at /project/third-party/faiss/faiss/impl/io.cpp:69: Error: 'f' failed: could not open /content/drive/My Drive/Colab_RAG_Project/chapters_db/Chapter_1/index.faiss for reading: No such file or directory
  [ERR] Chapter_2: Error in faiss::FileIOReader::FileIOReader(const char*) at /project/third-party/faiss/faiss/impl/io.cpp:69: Error: 'f' failed: could not open /content/drive/My Drive/Colab_RAG_Project/chapters_db/Chapter_2/index.faiss for reading: No such file or directory
  [ERR] Chapter_3: Error in faiss::FileIOReader::FileIOReader(const char*) at /project/third-party/faiss/faiss/impl/io.cpp:69: Error: 'f' failed: could not open /content/drive/My Drive/Colab_RAG_Project/chapters_db/Chapter_3/index.faiss for reading: No such file or directory
  [ERR] Chapter_4: Error in faiss::FileIOReader::FileIOReader(const char*) at /project/third-party/faiss/faiss/impl/io.cpp:69: Error: 'f' failed: could

In [4]:
import re

# --- HELPER TOOLS ---
def calculator_tool(expression):
    try:
        # Safety filter
        clean_expr = re.sub(r'[^0-9+\-*/().]', '', expression)
        return str(eval(clean_expr))
    except:
        return "Error in calculation"

# --- MAIN AGENT LOOP ---
def run_agent(user_query):
    # 1. DECISION STEP
    # We ask the model explicitly which tool to use
    prompt = f"""<|system|>
You are a Routing Agent. Analyze the user's question and choose the best tool.
- If the user asks about course concepts, definitions, or theory (RISC, Pipelining, Cycles), reply: TOOL: RAG [search query]
- If the user asks for a math calculation, reply: TOOL: CALC [math expression]
- If it is a general coding or chat question, reply: TOOL: NONE
<|end|>
<|user|>
{user_query}
<|end|>
<|assistant|>"""

    # Generate decision
    output = pipe(prompt, max_new_tokens=50)[0]['generated_text']

    # Parse decision
    context = ""
    tool_name = "General Chat"

    if "TOOL: RAG" in output:
        search_q = output.split("RAG")[1].strip()
        print(f"🔍 Searching Textbook for: '{search_q}'...")
        context = rag_tool.search(search_q)
        tool_name = "Textbook (RAG)"

    elif "TOOL: CALC" in output:
        math_expr = output.split("CALC")[1].strip()
        print(f"🧮 Calculating: '{math_expr}'...")
        result = calculator_tool(math_expr)
        context = f"The result of {math_expr} is {result}."
        tool_name = "Calculator"

    # 2. ANSWER STEP
    # Generate final answer using the context
    final_prompt = f"""<|system|>
You are a helpful teaching assistant. Use the provided context to answer the student.
If no context is provided, answer to the best of your ability.
<|end|>
<|user|>
Context:
{context}

Question:
{user_query}
<|end|>
<|assistant|>"""

    final_answer = pipe(final_prompt)[0]['generated_text']

    return f"\n----- [{tool_name}] -----\n{final_answer}\n-------------------------"

# --- TEST IT ---
print(run_agent("What is the difference between RISC and CISC?"))
print(run_agent("Calculate 50 * 20 + 5"))

The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.


🔍 Searching Textbook for: '[difference between RISC and CISC]

The difference between RISC (Reduced Instruction Set Computer) and CISC (Complex Instruction Set Computer) lies in their design'...

----- [Textbook (RAG)] -----
 The difference between Reduced Instruction Set Computer (RISC) and Complex Instruction Set Computer (CISC) lies in their approach to processing instructions.

RISC architecture is designed with a smaller set of simple and highly optimized instructions. The idea is that by simplifying the instructions, the processor can execute them more quickly and efficiently. RISC processors typically have a fixed instruction length, which simplifies the decoding process and allows for faster instruction execution. They often use a load/store architecture, where data is loaded into registers and stored back into memory.

CISC architecture, on the other hand, features a larger set of more complex instructions. These instructions can perform multiple operations within a single ins

In [5]:
  # Create an interactive chat loop
print("🤖 Agent is ready! Type 'exit' to stop.\n")

while True:
    # 1. Get input from you
    user_input = input("\nUser: ")

    # 2. Check if you want to quit
    if user_input.lower() in ["exit", "quit", "stop"]:
        print("Goodbye!")
        break

    # 3. Run the agent and print the response
    try:
        response = run_agent(user_input)
        print(response)
    except Exception as e:
        print(f"❌ An error occurred: {e}")

🤖 Agent is ready! Type 'exit' to stop.


User: hi

----- [General Chat] -----
 Hello! I'm Phi, your virtual teaching assistant. How can I help you today?
-------------------------

User: what is the cisc architecture ? 
🔍 Searching Textbook for: '[search query]

To provide the user with information about the CISC (Complex Instruction Set Computer) architecture, I would use the'...

----- [Textbook (RAG)] -----
 The CISC (Complex Instruction Set Computing) architecture refers to a type of computer processor design that is characterized by a large set of instructions that can execute several low-level operations within a single instruction. This contrasts with RISC (Reduced Instruction Set Computing) architecture, which uses a smaller set of instructions that are designed to execute simpler operations but often require multiple instructions to perform complex tasks.

CISC processors are designed to handle complex instructions that can perform multiple operations, such as memory access, a

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



----- [General Chat] -----
 Writing a Fibonacci sequence in assembly language can be quite specific to the processor architecture you're targeting (e.g., x86, ARM). Below is a simplified example of how you might write a Fibonacci sequence in x86 assembly language using NASM (Netwide Assembler). This code calculates the first 10 Fibonacci numbers and stores them in an array.

```asm
section.data
    fib_array dd 0, 1, 0, 0, 0, 0, 0, 0, 0, 0 ; Array to store Fibonacci numbers

section.bss
    counter resb 1

section.text
    global _start

_start:
    mov ecx, 10        ; Set counter for 10 Fibonacci numbers
    mov [counter], cl  ; Move counter value into counter variable

    mov eax, 0         ; First Fibonacci number
    mov [fib_array], eax
    mov ebx, 1         ; Second Fibonacci number
    mov [fib_array+4], ebx

    ; Calculate the next Fibonacci numbers
    calculate_fib:
        dec dword [counter]
        jz end_calculation

        mov edx, eax
        add edx, ebx
        